## Day 1: Search APIs for Fresh Information

A search API takes a text query and returns a list of results, each carrying a title, a snippet, and a
source URL. Below is a small **mock** search client — no network call, no API key — that matches the
shape a real client (e.g. a `requests.get(...)` call to a search provider's REST endpoint) would have.
Swapping the mock for a real client later only means changing the body of `search_web`, not any of its
callers.


In [ ]:
from dataclasses import dataclass

@dataclass
class SearchResult:
    title: str
    snippet: str
    source_url: str

# Mock local "index" standing in for a real search provider's backend.
# A real implementation would send `query` to an HTTP endpoint and parse JSON results instead.
_MOCK_INDEX = {
    "launch window definition": [
        SearchResult(
            "Launch Windows Explained",
            "A launch window is the time period during which a rocket can lift off to reach its target orbit.",
            "https://example-space.org/launch-windows",
        ),
        SearchResult(
            "Orbital Mechanics 101",
            "Launch windows are constrained by the relative positions of Earth and the destination body.",
            "https://example-space.org/orbital-mechanics",
        ),
    ],
    "reusable rocket landing methods": [
        SearchResult(
            "Booster Recovery Techniques",
            "Modern boosters land using either a controlled powered descent or a parachute-assisted splashdown.",
            "https://example-space.org/booster-recovery",
        ),
    ],
}

def search_web(query: str, max_results: int = 5) -> list[SearchResult]:
    """Mock search client. A real one would call a provider API and map its JSON to SearchResult."""
    key = query.lower().strip()
    return _MOCK_INDEX.get(key, [])[:max_results]

# A specific query hits the mock index; a vague one returns nothing useful.
good_hits = search_web("launch window definition")
vague_hits = search_web("space stuff")
print(f"specific query -> {len(good_hits)} results")
print(f"vague query -> {len(vague_hits)} results")


In [ ]:
def build_context_bundle(results: list[SearchResult]) -> str:
    """Combine search results into one string for an LLM prompt, keeping each snippet tied to its URL."""
    blocks = []
    for i, r in enumerate(results, start=1):
        blocks.append(f"[{i}] {r.title}\n{r.snippet}\nSource: {r.source_url}")
    return "\n\n".join(blocks)

bundle = build_context_bundle(good_hits)
print(bundle)


## Day 2: Embedding-Based Re-Ranking of Search Results

Reusing the hash-based bag-of-words embedding and cosine similarity from the earlier retrieval lesson,
we can re-rank raw search-API hits so the most relevant ones surface first. Below, five raw results for
a sourdough baking question come back in an arbitrary API order; re-ranking by similarity to the query
pulls the two truly relevant ones to the top.


In [ ]:
import hashlib
import re
from collections import Counter

VECTOR_SIZE = 64

def embed_text(text: str) -> list[float]:
    """Deterministic, fully local stand-in for a real embedding model call."""
    vector = [0.0] * VECTOR_SIZE
    words = re.findall(r"[a-z0-9]+", text.lower())
    for word, count in Counter(words).items():
        bucket = int(hashlib.md5(word.encode()).hexdigest(), 16) % VECTOR_SIZE
        vector[bucket] += count
    return vector

def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(y * y for y in b) ** 0.5
    return dot / (norm_a * norm_b) if norm_a and norm_b else 0.0


In [ ]:
query = "starter hydration ratio"

# Raw results as a search API might return them: relevance mixed with popularity.
raw_results = [
    SearchResult("History of Sourdough Bread", "This ancient bread-making technique dates back thousands of years.", "https://example-bake.org/history"),
    SearchResult("Hydration Ratio for Sourdough Starter", "A 100 percent hydration starter uses equal weights of flour and water.", "https://example-bake.org/hydration"),
    SearchResult("Best Bread Knives 2026", "A serrated knife makes cleaner slices through a crusty loaf.", "https://example-bake.org/knives"),
    SearchResult("Adjusting Starter Hydration for Climate", "Lower hydration starter mixtures ferment more slowly in humid kitchens.", "https://example-bake.org/climate-hydration"),
    SearchResult("Sourdough Discard Recipes", "Use leftover portions in pancakes or crackers instead of discarding them.", "https://example-bake.org/discard"),
]

def rerank_results(query: str, results: list[SearchResult], top_k: int = 2) -> list[SearchResult]:
    query_vec = embed_text(query)
    scored = [
        (cosine_similarity(query_vec, embed_text(r.title + " " + r.snippet)), r)
        for r in results
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [r for _, r in scored[:top_k]]

print("BEFORE (raw API order):")
for r in raw_results:
    print(f"  - {r.title}")

print("\nAFTER (embedding re-ranked, top 2):")
for r in rerank_results(query, raw_results, top_k=2):
    print(f"  - {r.title}")


## Day 3: Search + LLM Grounded Research

Grounding means the LLM answers only from retrieved material, cites which source backs each claim, and
admits when the material is insufficient rather than guessing. The pipeline below chains a search step,
a re-ranking step, and a (mock) grounded LLM call, using a network-protocol adoption question as the
running example.


In [ ]:
def build_grounded_prompt(question: str, bundle: str) -> str:
    return (
        "You are a research assistant. Answer using ONLY the material below.\n\n"
        f"MATERIAL:\n{bundle}\n\n"
        f"QUESTION: {question}\n\n"
        "Rules:\n"
        "- Cite the source number [n] after every factual claim.\n"
        "- If the material does not answer the question, say "
        "'Not enough information in the provided sources.'\n"
        "- Do not add outside knowledge.\n"
    )

def call_llm_stub(prompt: str) -> str:
    """Mock LLM call. A real client (e.g. an Anthropic Messages API call) would send `prompt`
    to a hosted model and return its text response instead of this canned reply."""
    return (
        "QUIC is now supported by default in several major browsers [1]. Adoption on the server "
        "side is growing but remains behind HTTP/2 in raw traffic share [2]. Not enough information "
        "in the provided sources to say when server-side adoption will overtake HTTP/2.\n\n"
        "Sources:\n[1] https://example-net.org/quic-browsers\n[2] https://example-net.org/quic-traffic-share"
    )


In [ ]:
protocol_index = {
    "quic protocol adoption": [
        SearchResult(
            "QUIC Support Across Browsers",
            "QUIC is enabled by default in several major browsers as of this year.",
            "https://example-net.org/quic-browsers",
        ),
        SearchResult(
            "Server-Side Traffic Share Report",
            "QUIC traffic is rising but still trails HTTP/2 in overall share among measured servers.",
            "https://example-net.org/quic-traffic-share",
        ),
    ],
}

def search_web_v2(query: str, max_results: int = 5) -> list[SearchResult]:
    return protocol_index.get(query.lower().strip(), [])[:max_results]

def research(question: str, searcher, ranker, summarizer) -> str:
    """Chain: search -> re-rank -> bundle with sources -> grounded LLM summary."""
    raw_hits = searcher(question)
    top_hits = ranker(question, raw_hits, top_k=2) if raw_hits else raw_hits
    bundle = build_context_bundle(top_hits)
    prompt = build_grounded_prompt(question, bundle)
    return summarizer(prompt)

answer = research(
    "quic protocol adoption",
    searcher=search_web_v2,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(answer)


## Day 4: AI Tool Evaluation Framework

Any specific tool name in this space goes stale fast, so the evaluation itself should stay
framework-agnostic: score a candidate tool on cost, security, and approval-friction, then reuse the
Day 3 `research()` pipeline to keep the underlying facts current. Below, a small scoring helper is
applied to two hypothetical products — a note-taking assistant and a spreadsheet copilot — to show the
checklist in code form.


In [ ]:
from dataclasses import dataclass, field

@dataclass
class ToolEvaluation:
    name: str
    pricing_model: str
    hidden_usage_costs: bool
    trains_on_input_data: bool
    sso_supported: bool
    requires_security_review: bool
    notes: str = ""

    def quick_verdict(self) -> str:
        """~10-minute first-pass verdict: green / yellow / red."""
        red_flags = sum([
            self.hidden_usage_costs,
            self.trains_on_input_data and not self.sso_supported,
        ])
        if red_flags == 0 and not self.requires_security_review:
            return "green"
        if red_flags >= 2:
            return "red"
        return "yellow"

candidates = [
    ToolEvaluation(
        name="note-taking assistant",
        pricing_model="per-seat monthly",
        hidden_usage_costs=False,
        trains_on_input_data=False,
        sso_supported=True,
        requires_security_review=False,
        notes="Opt-out training disclosed clearly in settings.",
    ),
    ToolEvaluation(
        name="spreadsheet copilot",
        pricing_model="usage-metered, billed to a connected API key",
        hidden_usage_costs=True,
        trains_on_input_data=True,
        sso_supported=False,
        requires_security_review=True,
        notes="Retention policy unclear; needs compliance sign-off before rollout.",
    ),
]

for c in candidates:
    print(f"{c.name}: verdict={c.quick_verdict()} | {c.notes}")


In [ ]:
# Tie back to Day 3: periodically research what's changed for a given tool category
# so the cost/security/approval-friction scoring above doesn't go stale.
landscape_index = {
    "spreadsheet copilot pricing and security changes": [
        SearchResult(
            "Spreadsheet Copilot Updates Data Retention Policy",
            "The vendor now offers an enterprise tier with opt-out training and a signed DPA.",
            "https://example-tools.org/copilot-policy-update",
        ),
    ],
}

def search_web_v3(query: str, max_results: int = 5) -> list[SearchResult]:
    return landscape_index.get(query.lower().strip(), [])[:max_results]

landscape_update = research(
    "spreadsheet copilot pricing and security changes",
    searcher=search_web_v3,
    ranker=rerank_results,
    summarizer=call_llm_stub,
)
print(landscape_update)
